In [1]:
import torch
import pandas as pd
from epiweeks import Week
import preprocess_data as prep
import model as mc
import matplotlib.pyplot as plt 


pd.options.mode.chained_assignment = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

regioes_estados = {
        'Sul': ['SC', 'PR', 'RS'],
        'Sudeste': ['SP', 'MG', 'RJ', 'ES'],
        'Nordeste': ['BA', 'CE', 'PE', 'PB', 'PI', 'RN', 'MA', 'AL', 'SE'],
        'Centro-Oeste': ['DF', 'MT', 'MS', 'GO'],
        'Norte': ['RO', 'AC', 'AM', 'RR', 'PA', 'AP', 'TO']
    } 
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

columns_to_normalize = ['casos','epiweek', 'biome', 'enso']

predict_n = 30
max_epiweek = 22
    
boxcox = False

TEST_YEAR = 2026


In [2]:
batch_size = 2
epochs = 250
media =True
verbose = 1
doenca = 'chikungunya'
min_delta = 0.0
patience= 25
lr = 2e-4
min_year = 2015
model_name = f'enso_22_26'

In [3]:
df = prep.load_cases_data(filename= f'./data/{doenca}.csv.gz')

enso = prep.load_enso_weekly()

enso_neutro = prep.load_enso_weekly(filename='data/enso_weekly_neutro.csv')

In [4]:
for region in regioes_estados.keys(): 

    print(region)

    label = f'{region}_{TEST_YEAR-1}_{model_name}'

    df_reg = df.loc[df.uf.isin(regioes_estados[region])]
    df_reg = df_reg.loc[df_reg.index >= pd.to_datetime(Week(2015,1).startdate())]


    X_train, y_train, X_future, norm_values, norm_enso =  prep.generate_regional_train_samples(df_reg, enso, 
                                                                                    TEST_YEAR,
                                                                                    max_epiweek = max_epiweek,
                                                                                    columns_to_normalize = columns_to_normalize,
                                                                                    min_year = min_year, 
                                                                                    boxcox = boxcox,
                                                                                    media = media)

    model = mc.LSTMWithFutureCovariatesV2(

                    hidden=64,

                    past_features=5,

                    future_cov_size=predict_n,

                    predict_n=predict_n,

                    dropout=0.2
                        )
    
    #model_path = f'./saved_models/trained_dengue_{region}_{TEST_YEAR-1}_enso_media.pt'
    #model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    #model.to(device)  
    
    trained_model, hist = mc.train_model(

            model=model,

            X_train=X_train,

            X_future=X_future,

            Y_train=y_train,

            label=label,

            batch_size=batch_size,

            epochs=epochs,

            lr=lr,

            patience=patience,

            verbose=verbose,

            doenca=doenca
)


Sul
Epoch 1/250 | Train: 0.6714 | Val: 0.6668
Epoch 2/250 | Train: 0.6521 | Val: 0.6417
Epoch 3/250 | Train: 0.6042 | Val: 0.5354
Epoch 4/250 | Train: 0.4281 | Val: 0.3367
Epoch 5/250 | Train: 0.2790 | Val: 0.2425
Epoch 6/250 | Train: 0.2164 | Val: 0.1909
Epoch 7/250 | Train: 0.1710 | Val: 0.1589
Epoch 8/250 | Train: 0.1504 | Val: 0.1368
Epoch 9/250 | Train: 0.1282 | Val: 0.1201
Epoch 10/250 | Train: 0.1128 | Val: 0.1074
Epoch 11/250 | Train: 0.1020 | Val: 0.0970
Epoch 12/250 | Train: 0.0925 | Val: 0.0886
Epoch 13/250 | Train: 0.0845 | Val: 0.0816
Epoch 14/250 | Train: 0.0771 | Val: 0.0756
Epoch 15/250 | Train: 0.0726 | Val: 0.0705
Epoch 16/250 | Train: 0.0682 | Val: 0.0662
Epoch 17/250 | Train: 0.0642 | Val: 0.0625
Epoch 18/250 | Train: 0.0624 | Val: 0.0592
Epoch 19/250 | Train: 0.0583 | Val: 0.0563
Epoch 20/250 | Train: 0.0552 | Val: 0.0536
Epoch 21/250 | Train: 0.0572 | Val: 0.0513
Epoch 22/250 | Train: 0.0510 | Val: 0.0491
Epoch 23/250 | Train: 0.0473 | Val: 0.0472
Epoch 24/250 | T